# Test de Data Wrangler

Ce notebook contient un jeu de donnees volontairement imparfait. Executez la cellule suivante, puis utilisez **Open 'Data Wrangler'** sur le DataFrame `customers` depuis l'explorateur de variables ou la sortie de cellule.

Cas a tester : filtrage, tri, suppression des doublons, gestion des valeurs manquantes, conversion de types et normalisation de texte.

In [2]:
import pandas as pd

customers = pd.DataFrame({
    'customer_id': [101, 102, 103, 104, 104, 105, 106],
    'name': ['Alice Martin', 'Benoit Durand', 'Chloe Petit', 'David Leroy', 'David Leroy', 'Emma Roy', 'Farid Morel'],
    'city': ['Paris', 'lyon', 'PARIS', None, None, 'Marseille', 'marseille'],
    'age': ['29', '34', None, '41', '41', 'unknown', '27'],
    'signup_date': ['2025-01-12', '2025/02/03', '12-03-2025', None, None, '2025-04-22', '2025-05-18'],
    'revenue': ['125,50', '980,00', '45,90', '1200,00', '1200,00', None, '75,25'],
    'active': [True, True, False, True, True, None, False]
})

customers

,customer_id,name,city,age,signup_date,revenue,active
0,101,Alice Martin,Paris,29,2025-01-12,"125,50",True
1,102,Benoit Durand,lyon,34,2025/02/03,"980,00",True
2,103,Chloe Petit,PARIS,None,12-03-2025,"45,90",False
3,104,David Leroy,None,41,None,"1200,00",True
4,104,David Leroy,None,41,None,"1200,00",True
5,105,Emma Roy,Marseille,unknown,2025-04-22,None,None
6,106,Farid Morel,marseille,27,2025-05-18,"75,25",False


In [3]:
# Apercu de la structure, utile pour le premier test dans Data Wrangler
customers.info()
customers.describe(include='all')

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7 entries, 0 to 6
Data columns (total 7 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   customer_id  7 non-null      int64 
 1   name         7 non-null      object
 2   city         5 non-null      object
 3   age          6 non-null      object
 4   signup_date  5 non-null      object
 5   revenue      6 non-null      object
 6   active       6 non-null      object
dtypes: int64(1), object(6)
memory usage: 520.0+ bytes


,customer_id,name,city,age,signup_date,revenue,active
count,7.000000,7,5,6,5,6,6
unique,NaN,6,5,5,5,5,2
top,NaN,David Leroy,Paris,41,2025-01-12,"1200,00",True
freq,NaN,2,1,2,1,2,4
mean,103.571429,NaN,NaN,NaN,NaN,NaN,NaN
std,1.718249,NaN,NaN,NaN,NaN,NaN,NaN
min,101.000000,NaN,NaN,NaN,NaN,NaN,NaN
25%,102.500000,NaN,NaN,NaN,NaN,NaN,NaN
50%,104.000000,NaN,NaN,NaN,NaN,NaN,NaN
75%,104.500000,NaN,NaN,NaN,NaN,NaN,NaN


## Nettoyage de reference

Cette version sert a comparer les transformations realisees manuellement dans Data Wrangler avec un resultat attendu.

In [4]:
clean_customers = customers.copy()

clean_customers = clean_customers.drop_duplicates()
clean_customers['city'] = clean_customers['city'].str.strip().str.title()
clean_customers['age'] = pd.to_numeric(clean_customers['age'], errors='coerce')
clean_customers['revenue'] = (
    clean_customers['revenue']
    .str.replace(',', '.', regex=False)
    .pipe(pd.to_numeric, errors='coerce')
)
clean_customers['signup_date'] = pd.to_datetime(
    clean_customers['signup_date'], errors='coerce', format='mixed', dayfirst=True
)
clean_customers['age'] = clean_customers['age'].fillna(clean_customers['age'].median())

clean_customers

,customer_id,name,city,age,signup_date,revenue,active
0,101,Alice Martin,Paris,29.0,2025-01-12,125.50,True
1,102,Benoit Durand,Lyon,34.0,2025-02-03,980.00,True
2,103,Chloe Petit,Paris,31.5,2025-03-12,45.90,False
3,104,David Leroy,None,41.0,NaT,1200.00,True
5,105,Emma Roy,Marseille,31.5,2025-04-22,NaN,None
6,106,Farid Morel,Marseille,27.0,2025-05-18,75.25,False


In [5]:
clean_customers.info()
clean_customers.isna().sum()

<class 'pandas.core.frame.DataFrame'>
Index: 6 entries, 0 to 6
Data columns (total 7 columns):
 #   Column       Non-Null Count  Dtype         
---  ------       --------------  -----         
 0   customer_id  6 non-null      int64         
 1   name         6 non-null      object        
 2   city         5 non-null      object        
 3   age          6 non-null      float64       
 4   signup_date  5 non-null      datetime64[ns]
 5   revenue      5 non-null      float64       
 6   active       5 non-null      object        
dtypes: datetime64[ns](1), float64(2), int64(1), object(3)
memory usage: 384.0+ bytes


customer_id    0
name           0
city           1
age            0
signup_date    1
revenue        1
active         1
dtype: int64